# FaceProof 04 — robustness sweep (RQ2)

**Day 4.** Corrupt the **test images only**, re-extract features through the frozen backbones,
predict with the saved probes (nb 03), plot AUROC vs strength. JPEG is core; others extension.
Strengths from `configs/experiment.yaml`.

## 1. Setup (GPU on — we re-extract features)

In [ ]:
REPO_URL = "https://github.com/Ezed9/faceProof.git"
BRANCH   = "main"
!git clone -b $BRANCH $REPO_URL
%cd faceProof
!pip install -q open_clip_torch scikit-learn joblib pyyaml
import sys, os
sys.path.append(os.getcwd())

In [ ]:
from google.colab import drive
from pathlib import Path
import numpy as np, pandas as pd
drive.mount('/content/drive')
ROOT          = Path('/content/drive/MyDrive/faceproof')
CROPS_ROOT    = ROOT / 'data' / 'crops'
FEATURES_ROOT = ROOT / 'models' / 'features'
PROBES_ROOT   = ROOT / 'models' / 'probes'
REPORTS_ROOT  = ROOT / 'reports'
MANIFEST      = ROOT / 'data' / 'manifest.csv'
RESULTS_CSV   = REPORTS_ROOT / 'results.csv'
(REPORTS_ROOT / 'figures').mkdir(parents=True, exist_ok=True)
PROBES_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
import csv
RESULT_FIELDS = ['condition','train_gen','test_gen','corruption','strength','seed','metric','value']
def log_result(**row):
    full = {k: row.get(k, '') for k in RESULT_FIELDS}
    new = not RESULTS_CSV.exists()
    with open(RESULTS_CSV, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=RESULT_FIELDS)
        if new: w.writeheader()
        w.writerow(full)

## 2. Load manifest, probes, corruption config

> **Why these masks:** the unseen SD rows live in `split=='test'` and are *all fake*, so AUROC on that split alone is undefined. Each eval group pairs the held-out **reals** (label 0 in `test_indist`) with the fakes of interest: in-dist = StyleGAN, cross-gen = Stable Diffusion.

In [ ]:
import yaml, joblib
df=pd.read_csv(MANIFEST); y=df['label'].values
probes={c: joblib.load(PROBES_ROOT/f'{c}.joblib') for c in ['c1_resnet','c4_clip']}
exp=yaml.safe_load(open('configs/experiment.yaml'))['corruptions']
mcfg=yaml.safe_load(open('configs/model.yaml'))['clip']
real_test=(df['label']==0)&(df['split']=='test_indist')
GROUPS={'stylegan_indist': (df['split']=='test_indist'),
        'stable_diffusion': (real_test | (df['generator']=='stable_diffusion'))}
print({k:int(v.sum()) for k,v in GROUPS.items()})
for k,v in GROUPS.items():
    yy=y[v.values]; assert yy.min()==0 and yy.max()==1, f'{k} needs both classes'

## 3. Helper: corrupt test images → re-extract features

Corrupted copies are written as **PNG** (lossless) so the only lossy step is the corruption
itself — no extra JPEG re-encode confound. We assert the count is preserved (alignment).

In [ ]:
import shutil
from PIL import Image
from src import corruptions
from src.features import extract_clip, extract_resnet
TMP=Path('/content/_corrupt')
def corrupt_and_extract(paths, corr_name, strength):
    if TMP.exists(): shutil.rmtree(TMP)
    TMP.mkdir(parents=True)
    fn=corruptions.CORRUPTIONS[corr_name]; out=[]
    for i,p in enumerate(paths):
        img=fn(Image.open(p).convert('RGB'), strength)
        op=TMP/f'{i:06d}.png'; img.save(op,'PNG'); out.append(str(op))
    assert len(out)==len(paths), 'corruption dropped images -> labels would misalign'
    Xc=extract_clip(out, backbone=mcfg['backbone'], pretrained=mcfg['pretrained'], batch_size=64, cache=None)
    Xr=extract_resnet(out, batch_size=64, cache=None)
    return {'c4_clip':Xc,'c1_resnet':Xr}

## 4. Run the sweep (JPEG core; uncomment extensions later)

In [ ]:
from src.metrics import auroc
from src.probe import predict_proba
SWEEPS={'jpeg':('jpeg',exp['jpeg_quality']),
        # 'resize':('resize',exp['resize_factor']),
        # 'blur':('blur',exp['blur_sigma']),
        # 'noise':('noise',exp['noise_std']),
       }
curves=[]
for label,(corr,strengths) in SWEEPS.items():
    for tg,mask in GROUPS.items():
        paths=df.loc[mask,'path'].tolist(); yy=y[mask.values]
        for s in strengths:
            feats=corrupt_and_extract(paths,corr,s)
            for cond,clf in probes.items():
                au=auroc(yy,predict_proba(clf,feats[cond]))
                log_result(condition=cond,train_gen='stylegan',test_gen=tg,corruption=corr,strength=s,seed=13,metric='AUROC',value=round(au,4))
                curves.append({'cond':cond,'test_gen':tg,'corr':corr,'strength':s,'AUROC':round(au,4)})
            print(f'{corr}={s} {tg}: done')
cur=pd.DataFrame(curves); print(cur.to_string(index=False))

## 5. ✅ Gate — JPEG AUROC-vs-quality curve for C1 and C4

In [ ]:
import matplotlib.pyplot as plt
jp=cur[cur['corr']=='jpeg']
fig,ax=plt.subplots(figsize=(7,4.5))
for (cond,tg),g in jp.groupby(['cond','test_gen']):
    g=g.sort_values('strength',ascending=False)
    ax.plot(g['strength'],g['AUROC'],marker='o',label=f'{cond}/{tg}')
ax.invert_xaxis(); ax.set_xlabel('JPEG quality (lower = more compression)'); ax.set_ylabel('AUROC')
ax.set_title('Robustness: AUROC vs JPEG quality'); ax.legend(fontsize=8); ax.grid(alpha=.3)
fig.savefig(REPORTS_ROOT/'figures'/'robustness_jpeg.png',dpi=150,bbox_inches='tight'); plt.show()
print('✅ Day 4 gate: JPEG curve saved + rows in results.csv')